# Train PINN on euclid-like dataset
This notebook shows how to set up the PINN training loop. The PINN uses a
reprojected physics loss to enforce consistency with observed spectrograms.
This expanded notebook includes data inspection, a short training run, and
visualisations of the physics residuals (A(y_pred) − x_obs).

In [ ]:
# imports
import torch
from spectangle.utils.training import get_device
from torch.utils.data import DataLoader, Dataset
import numpy as np
from spectangle.simulations.io import load_simulation
from spectangle.models.pinn import PINN
import matplotlib.pyplot as plt

DEVICE = get_device()
print(f'Using device: {DEVICE}')

class H5ComplexDataset(Dataset):
    def __init__(self, path):
        data = load_simulation(path)
        self.samples = data['samples']
        meta = data['metadata']
        self.pad_y = int(meta['pad_y'])
        self.pad_x = int(meta['pad_x'])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        spec = s['spectrograms'].astype(np.float32)
        cube = s['cube'].astype(np.float32)
        direct = s.get('direct_image')
        if direct is not None:
            x = np.concatenate([spec, direct[None, ...]], axis=0)
        else:
            x = spec
        return torch.from_numpy(x), torch.from_numpy(cube)

# instantiate
path = 'data/raw/complex_euclid_50s.h5'
ds = H5ComplexDataset(path)
loader = DataLoader(ds, batch_size=2, shuffle=True)

# instantiate PINN
pinn = PINN(in_channels=4, n_lambda=128, image_shape=(128,128), lambda_physics=1.0)
opt = torch.optim.Adam(pinn.parameters(), lr=5e-5)
pinn = pinn.to(DEVICE)

# quick look at one sample
x0, y0 = ds[0]
print('x0 shape (C,H,W):', x0.shape)
print('y0 shape (n_lambda, ny, nx):', y0.shape)

# Visualise observed spectrograms vs ground-truth broadband
specs0 = x0.numpy()[:4]
fig, ax = plt.subplots(1, 5, figsize=(16, 3))
for k in range(4):
    ax[k].imshow(specs0[k], origin='lower', cmap='inferno')
    ax[k].set_title(f'Obs spec {k+1}')
    ax[k].axis('off')
ax[4].imshow(y0.sum(axis=0), origin='lower', cmap='gray')
ax[4].set_title('GT broadband')
ax[4].axis('off')
plt.tight_layout()
plt.show()

## Short training run with physics diagnostics
Run a few epochs and record the reconstruction and physics losses. After
training, reproject a predicted cube using the PINN's internal differentiable
forward model and visualise the physics residuals.

In [ ]:
# tiny training run
loss_history = []
phys_history = []
rec_history = []
for epoch in range(3):
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        cube_pred, phys_loss, rec_loss = pinn.forward_with_physics_loss(x, y)
        loss = rec_loss + pinn.lambda_physics * phys_loss
        opt.zero_grad()
        loss.backward()
        opt.step()
    loss_history.append(loss.item())
    phys_history.append(phys_loss.item())
    rec_history.append(rec_loss.item())
    print(f'Epoch {epoch} loss={loss.item():.4f} (rec={rec_loss.item():.4f} phys={phys_loss.item():.4f})')

# Plot loss curves
plt.figure(figsize=(6, 3))
plt.plot(loss_history, label='total')
plt.plot(rec_history, label='rec')
plt.plot(phys_history, label='physics')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training losses (toy run)')
plt.tight_layout()
plt.show()

## Visualise reprojected spectrograms and residuals
Take a sample, predict a cube, reproject it with the differentiable forward
model inside the PINN and compare to the observed spectrograms. Plot the
observed spectrograms, reprojected spectrograms and their difference to
assess where physics residuals are large.

In [ ]:
# Visualisation of reprojected spectrograms
pinn.eval()
with torch.no_grad():
    x_sample, y_sample = ds[0]
    x_in = x_sample.unsqueeze(0).to(DEVICE)
    cube_pred = pinn(x_in)  # (B, n_lambda, ny, nx)
    reprojected = pinn.physics_model.forward(cube_pred)

obs = x_in[0, :4].cpu().numpy()
rep = reprojected[0].cpu().numpy()
res = rep - obs

fig, axs = plt.subplots(3, 4, figsize=(14, 8))
for k in range(4):
    axs[0, k].imshow(obs[k], origin='lower', cmap='inferno')
    axs[0, k].set_title(f'Obs spec {k+1}')
    axs[0, k].axis('off')

    axs[1, k].imshow(rep[k], origin='lower', cmap='inferno')
    axs[1, k].set_title(f'Reprojected {k+1}')
    axs[1, k].axis('off')

    im = axs[2, k].imshow(res[k], origin='lower', cmap='bwr', vmin=-np.max(np.abs(res)), vmax=np.max(np.abs(res)))
    axs[2, k].set_title(f'Residual {k+1}')
    axs[2, k].axis('off')
    fig.colorbar(im, ax=axs[2, k], fraction=0.03)

plt.suptitle('Observed / Reprojected / Residual (sample 0)')
plt.tight_layout()
plt.show()

Next steps and suggestions:
- Run a longer training with checkpointing and validation.
- Save periodic examples of reprojected spectrograms to monitor physics loss.
- Experiment with lambda_physics in the config YAML files to balance data and
  physics terms.